# Database Setup and Data Loading

This notebook sets up the PostgreSQL database and loads the customer behavioral data.

## Step 1: Start PostgreSQL Container

In [ ]:
# Start PostgreSQL using Docker Compose
# Run this in terminal: docker-compose up -d
print("Run 'docker-compose up -d' in the project directory to start PostgreSQL")

## Step 2: Create Schema and Tables

In [ ]:
import os
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

DB_PARAMS = {
    "host": os.environ["POSTGRES_HOST"],
    "port": os.environ.get("POSTGRES_PORT", "5432"),
    "database": os.environ["POSTGRES_DB"],
    "user": os.environ["POSTGRES_USER"],
    "password": os.environ["POSTGRES_PASSWORD"],
    "sslmode": os.environ.get("POSTGRES_SSLMODE", "require"),
}
print("Connected settings loaded from .env (password hidden)")
print("host set:", bool(DB_PARAMS["host"]))
print("database:", DB_PARAMS["database"])


## Step 3: Load Data into Staging Tables

In [ ]:
def load_csv_to_staging(csv_path, table_name):
    """Load CSV data into staging table"""
    df = pd.read_csv(csv_path)
    
    conn = psycopg2.connect(**DB_PARAMS)
    cursor = conn.cursor()
    
    # Create column list and placeholders
    columns = df.columns.tolist()
    columns_lower = [col.lower() for col in columns]
    placeholders = ', '.join(['%s'] * len(columns))
    column_names = ', '.join(columns_lower)
    
    insert_query = f"INSERT INTO staging.{table_name} ({column_names}) VALUES ({placeholders})"
    
    try:
        # Convert DataFrame to list of tuples
        data = [tuple(row) for row in df.values]
        
        # Execute batch insert
        cursor.executemany(insert_query, data)
        conn.commit()
        print(f"Loaded {len(df)} rows into staging.{table_name}")
    except Exception as e:
        conn.rollback()
        print(f"Error loading data into {table_name}: {e}")
    finally:
        cursor.close()
        conn.close()

# Load customer_info
customer_info_path = r"D:\Ahmed\My Projects\telecom intelligence project\data\processed\customer_info_db.csv"
load_csv_to_staging(customer_info_path, 'customer_info_staging')

# Load usage_data
usage_data_path = r"D:\Ahmed\My Projects\telecom intelligence project\data\processed\usage_data_db.csv"
load_csv_to_staging(usage_data_path, 'usage_data_staging')

# Load churn_labels
churn_labels_path = r"D:\Ahmed\My Projects\telecom intelligence project\data\processed\churn_labels_db.csv"
load_csv_to_staging(churn_labels_path, 'churn_labels_staging')

## Step 4: Transform and Load to Core Tables

In [ ]:
def execute_procedure(procedure_name):
    """Execute a stored procedure"""
    conn = psycopg2.connect(**DB_PARAMS)
    cursor = conn.cursor()
    
    try:
        cursor.execute(f"CALL {procedure_name}()")
        conn.commit()
        print(f"Successfully executed procedure: {procedure_name}")
    except Exception as e:
        conn.rollback()
        print(f"Error executing procedure {procedure_name}: {e}")
    finally:
        cursor.close()
        conn.close()

# Load data from staging to core
execute_procedure('load_customer_data')

## Step 5: Verify Data Loading

In [ ]:
def execute_query(query):
    """Execute a query and return results as DataFrame"""
    conn = psycopg2.connect(**DB_PARAMS)
    
    try:
        df = pd.read_sql_query(query, conn)
        return df
    finally:
        conn.close()

# Check customer count
customer_count = execute_query("SELECT COUNT(*) as count FROM core.customers")
print(f"Customers in core: {customer_count['count'].iloc[0]}")

# Check usage records
usage_count = execute_query("SELECT COUNT(*) as count FROM core.customer_usage")
print(f"Usage records in core: {usage_count['count'].iloc[0]}")

# Check churn labels
churn_count = execute_query("SELECT COUNT(*) as count FROM core.churn_labels")
print(f"Churn labels in core: {churn_count['count'].iloc[0]}")

## Step 6: Data quality checks

Uses the connection from Step 2 (credentials from .env only).


In [ ]:
# Check for customers without usage data
no_usage = execute_query("""
    SELECT c.customer_id 
    FROM core.customers c
    LEFT JOIN core.customer_usage u ON c.customer_id = u.customer_id
    WHERE u.customer_id IS NULL
    LIMIT 5
""")
print(f"Customers without usage data: {len(no_usage)}")

# Check for customers without churn labels
no_churn = execute_query("""
    SELECT c.customer_id 
    FROM core.customers c
    LEFT JOIN core.churn_labels ch ON c.customer_id = ch.customer_id
    WHERE ch.customer_id IS NULL
    LIMIT 5
""")
print(f"Customers without churn labels: {len(no_churn)}")

# Data quality summary
quality_summary = execute_query("""
    SELECT
        COUNT(*) as total_customers,
        COUNT(DISTINCT customer_id) as unique_customers,
        ROUND(AVG(age), 2) as avg_age,
        ROUND(AVG(monthly_charges), 2) as avg_charges,
        SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) as missing_age
    FROM core.customers
""")
print("\nData Quality Summary:")
print(quality_summary)

## Step 7: Clear Staging Tables (Optional)

In [ ]:
# Clear staging tables after successful load
execute_procedure('clear_staging')

## Next Steps

After successful database setup:
1. Implement SQL analytics queries
2. Perform EDA
3. Build ML models
4. Create visualizations